In [15]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Avenir"

In [16]:
data_path = Path("../data/processed/clean_womens_shoes.parquet")

shoes = pd.read_parquet(data_path)

print("Dataset shape:", shoes.shape)
shoes.head()

Dataset shape: (53176, 10)


,train_id,name,item_condition_id,category_name,brand_name,price,shipping,item_description,shoe_type,condition_group
0,14,HOLD for Dogs2016 Minnetonka boots,3,Women/Shoes/Boots,UGG Australia,43.0,0,Authentic. Suede fringe boots. Great condition...,Boots,Condition 3
1,70,Adidas Ultraboost Shoes,3,Women/Shoes/Athletic,Adidas,61.0,0,Overall good condition. A few signs of wear,Athletic,Condition 3
2,107,Boots NWT 6.5,1,Women/Shoes/Boots,Merona,13.0,1,Merona short boot new with tag size 6.5 come j...,Boots,Condition 1
3,108,New Duck Boots sz.7.5,1,Women/Shoes/Boots,Boulevard Boutique,38.0,0,New Duck Boots Sz.7.5 Stock up on These Trendy...,Boots,Condition 1
4,115,Steve Madden wedges,2,Women/Shoes/Pumps,Steve Madden,25.0,1,Never worn!!! Brown leather strap wedges!,Pumps,Condition 2


In [17]:
def bootstrap_retention_interval(
    condition_1_prices,
    condition_3_prices,
    n_bootstrap=2000,
    confidence_level=0.95,
    random_state=42
):
    """
    Calculate Condition 3 price retention and a bootstrap confidence interval.
    """

    condition_1_prices = np.asarray(condition_1_prices)
    condition_3_prices = np.asarray(condition_3_prices)

    if len(condition_1_prices) < 10 or len(condition_3_prices) < 10:
        return {
            "retention_percent": np.nan,
            "ci_lower": np.nan,
            "ci_upper": np.nan
        }

    retention_percent = (
        np.median(condition_3_prices)
        / np.median(condition_1_prices)
        * 100
    )

    rng = np.random.default_rng(random_state)

    bootstrap_results = []

    for _ in range(n_bootstrap):
        condition_1_sample = rng.choice(
            condition_1_prices,
            size=len(condition_1_prices),
            replace=True
        )

        condition_3_sample = rng.choice(
            condition_3_prices,
            size=len(condition_3_prices),
            replace=True
        )

        sample_retention = (
            np.median(condition_3_sample)
            / np.median(condition_1_sample)
            * 100
        )

        bootstrap_results.append(sample_retention)

    alpha = 1 - confidence_level

    ci_lower = np.percentile(
        bootstrap_results,
        100 * alpha / 2
    )

    ci_upper = np.percentile(
        bootstrap_results,
        100 * (1 - alpha / 2)
    )

    return {
        "retention_percent": retention_percent,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper
    }

In [18]:
nike_athletic = shoes[
    (shoes["brand_name"] == "Nike")
    & (shoes["shoe_type"] == "Athletic")
].copy()

nike_condition_1_prices = nike_athletic.loc[
    nike_athletic["condition_group"] == "Condition 1",
    "price"
]

nike_condition_3_prices = nike_athletic.loc[
    nike_athletic["condition_group"] == "Condition 3",
    "price"
]

nike_bootstrap_result = bootstrap_retention_interval(
    condition_1_prices=nike_condition_1_prices,
    condition_3_prices=nike_condition_3_prices
)

nike_bootstrap_result

{'retention_percent': np.float64(48.38709677419355),
 'ci_lower': np.float64(46.15384615384615),
 'ci_upper': np.float64(49.18032786885246)}

In [19]:
nike_summary = (
    nike_athletic
    .groupby("condition_group")["price"]
    .agg(["count", "median", "mean", "min", "max"])
)

nike_summary

,count,median,mean,min,max
condition_group,,,,,
Condition 1,1691,62.0,65.756357,5.0,381.0
Condition 2,1919,41.0,46.126628,5.0,215.0
Condition 3,3751,30.0,33.786990,3.0,299.0
Condition 4–5,473,19.0,20.672304,6.0,58.0


In [20]:
condition_1_median = nike_condition_1_prices.median()
condition_3_median = nike_condition_3_prices.median()

print("Condition 1 median:", condition_1_median)
print("Condition 3 median:", condition_3_median)
print(
    "Retention:",
    condition_3_median / condition_1_median * 100
)

Condition 1 median: 62.0
Condition 3 median: 30.0
Retention: 48.38709677419355


In [21]:
nike_result_summary = pd.DataFrame({
    "brand_name": ["Nike"],
    "shoe_type": ["Athletic"],
    "condition_1_count": [len(nike_condition_1_prices)],
    "condition_3_count": [len(nike_condition_3_prices)],
    "condition_1_median_price": [nike_condition_1_prices.median()],
    "condition_3_median_price": [nike_condition_3_prices.median()],
    "retention_percent": [
        nike_bootstrap_result["retention_percent"]
    ],
    "ci_lower": [
        nike_bootstrap_result["ci_lower"]
    ],
    "ci_upper": [
        nike_bootstrap_result["ci_upper"]
    ]
})

nike_result_summary.round(2)

,brand_name,shoe_type,condition_1_count,condition_3_count,condition_1_median_price,condition_3_median_price,retention_percent,ci_lower,ci_upper
0,Nike,Athletic,1691,3751,62.0,30.0,48.39,46.15,49.18


In [22]:
bootstrap_rows = []

grouped_profiles = shoes.groupby(
    ["brand_name", "shoe_type"]
)

for (brand_name, shoe_type), profile in grouped_profiles:

    condition_1_prices = profile.loc[
        profile["condition_group"] == "Condition 1",
        "price"
    ]

    condition_3_prices = profile.loc[
        profile["condition_group"] == "Condition 3",
        "price"
    ]

    if len(condition_1_prices) < 10 or len(condition_3_prices) < 10:
        continue

    retention_result = bootstrap_retention_interval(
        condition_1_prices=condition_1_prices,
        condition_3_prices=condition_3_prices
    )

    condition_1_median_result = bootstrap_median_interval(
        condition_1_prices
    )

    condition_3_median_result = bootstrap_median_interval(
        condition_3_prices
    )

    bootstrap_rows.append({
        "brand_name": brand_name,
        "shoe_type": shoe_type,

        "condition_1_count": len(condition_1_prices),
        "condition_3_count": len(condition_3_prices),

        "condition_1_median_price":
            condition_1_median_result["median_price"],

        "condition_1_median_ci_lower":
            condition_1_median_result["ci_lower"],

        "condition_1_median_ci_upper":
            condition_1_median_result["ci_upper"],

        "condition_3_median_price":
            condition_3_median_result["median_price"],

        "condition_3_median_ci_lower":
            condition_3_median_result["ci_lower"],

        "condition_3_median_ci_upper":
            condition_3_median_result["ci_upper"],

        "retention_percent":
            retention_result["retention_percent"],

        "retention_ci_lower":
            retention_result["ci_lower"],

        "retention_ci_upper":
            retention_result["ci_upper"]
    })

bootstrap_results = pd.DataFrame(bootstrap_rows)

print(
    "Qualifying brand-shoe profiles:",
    len(bootstrap_results)
)

bootstrap_results.head()

Qualifying brand-shoe profiles: 133


,brand_name,shoe_type,condition_1_count,condition_3_count,condition_1_median_price,condition_1_median_ci_lower,condition_1_median_ci_upper,condition_3_median_price,condition_3_median_ci_lower,condition_3_median_ci_upper,retention_percent,retention_ci_lower,retention_ci_upper
0,ALDO,Boots,13,52,46.0,28.0,56.0,23.0,21.5,27.0,50.000000,39.285714,87.500000
1,ALDO,Pumps,11,78,31.0,23.0,36.0,20.0,18.0,23.0,64.516129,51.388889,86.956522
2,ASICS,Athletic,42,184,48.0,41.0,56.0,24.0,22.0,26.0,50.000000,40.677966,60.000000
3,Adidas,Athletic,454,352,106.0,92.5,126.0,33.0,30.0,36.0,31.132075,24.997490,36.985539
4,Adidas,Fashion Sneakers,203,271,66.0,64.0,71.0,41.0,39.0,44.0,62.121212,56.060606,67.213115


In [23]:
output_path = Path(
    "../data/processed/brand_shoe_bootstrap_intervals.parquet"
)

bootstrap_results.to_parquet(
    output_path,
    index=False
)

print("Updated file saved to:", output_path)
print("Dataset shape:", bootstrap_results.shape)
print("Columns:")
print(bootstrap_results.columns.tolist())

Updated file saved to: ../data/processed/brand_shoe_bootstrap_intervals.parquet
Dataset shape: (133, 13)
Columns:
['brand_name', 'shoe_type', 'condition_1_count', 'condition_3_count', 'condition_1_median_price', 'condition_1_median_ci_lower', 'condition_1_median_ci_upper', 'condition_3_median_price', 'condition_3_median_ci_lower', 'condition_3_median_ci_upper', 'retention_percent', 'retention_ci_lower', 'retention_ci_upper']


In [24]:
def bootstrap_median_interval(
    prices,
    n_bootstrap=2000,
    confidence_level=0.95,
    random_state=42
):
    """
    Calculate a median price and its bootstrap confidence interval.
    """

    prices = np.asarray(prices)

    if len(prices) < 10:
        return {
            "median_price": np.nan,
            "ci_lower": np.nan,
            "ci_upper": np.nan
        }

    median_price = np.median(prices)

    rng = np.random.default_rng(random_state)

    bootstrap_medians = []

    for _ in range(n_bootstrap):
        sample = rng.choice(
            prices,
            size=len(prices),
            replace=True
        )

        bootstrap_medians.append(
            np.median(sample)
        )

    alpha = 1 - confidence_level

    ci_lower = np.percentile(
        bootstrap_medians,
        100 * alpha / 2
    )

    ci_upper = np.percentile(
        bootstrap_medians,
        100 * (1 - alpha / 2)
    )

    return {
        "median_price": median_price,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper
    }

In [25]:
nike_condition_1_median_result = bootstrap_median_interval(
    nike_condition_1_prices
)

nike_condition_3_median_result = bootstrap_median_interval(
    nike_condition_3_prices
)

print("Condition 1:")
print(nike_condition_1_median_result)

print("\nCondition 3:")
print(nike_condition_3_median_result)

Condition 1:
{'median_price': np.float64(62.0), 'ci_lower': np.float64(61.0), 'ci_upper': np.float64(65.0)}

Condition 3:
{'median_price': np.float64(30.0), 'ci_lower': np.float64(29.0), 'ci_upper': np.float64(30.0)}
